# EDA — Binary Diabetes Dataset
**Dataset:** `diabetes_binary_health_indicators_BRFSS2015.csv`  
**Source:** CDC Behavioral Risk Factor Surveillance System (BRFSS) 2015  
**Target:** `Diabetes_binary` — 0 = No diabetes, 1 = Prediabetes or diabetes  

**Goal:** Understand the data structure, distributions, class imbalance, and key feature relationships before modeling.

## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100

DATA_PATH = '../data/raw/diabetes_binary.csv'

## 1. Load & Basic Inspection

In [ ]:
df = pd.read_csv(DATA_PATH)
print(f'Shape: {df.shape}')
df.head()

In [ ]:
df.info()

In [ ]:
df.describe().T

### Feature Reference

| Feature | Type | Description |
|---|---|---|
| `Diabetes_binary` | Binary Target | 0 = no diabetes, 1 = prediabetes/diabetes |
| `HighBP` | Binary | High blood pressure |
| `HighChol` | Binary | High cholesterol |
| `CholCheck` | Binary | Cholesterol check in past 5 years |
| `BMI` | Integer | Body Mass Index |
| `Smoker` | Binary | Smoked ≥100 cigarettes lifetime |
| `Stroke` | Binary | Ever had a stroke |
| `HeartDiseaseorAttack` | Binary | Coronary heart disease or MI |
| `PhysActivity` | Binary | Physical activity in past 30 days |
| `Fruits` | Binary | Fruit consumption ≥1x/day |
| `Veggies` | Binary | Vegetable consumption ≥1x/day |
| `HvyAlcoholConsump` | Binary | Heavy alcohol use |
| `AnyHealthcare` | Binary | Has healthcare coverage |
| `NoDocbcCost` | Binary | Couldn't see doctor due to cost |
| `GenHlth` | Ordinal (1–5) | General health (1=excellent, 5=poor) |
| `MentHlth` | Integer (0–30) | Poor mental health days (past 30) |
| `PhysHlth` | Integer (0–30) | Poor physical health days (past 30) |
| `DiffWalk` | Binary | Difficulty walking/climbing stairs |
| `Sex` | Binary | 0=female, 1=male |
| `Age` | Ordinal (1–13) | Age category (1=18–24, 13=80+) |
| `Education` | Ordinal (1–6) | Education level |
| `Income` | Ordinal (1–8) | Household income level |

## 2. Data Quality

In [ ]:
# Missing values
missing = df.isnull().sum()
print('Missing values per column:')
print(missing[missing > 0] if missing.any() else 'No missing values — dataset is clean.')

In [ ]:
# Duplicate rows
n_dupes = df.duplicated().sum()
print(f'Duplicate rows: {n_dupes} ({n_dupes / len(df) * 100:.2f}%)')

We removed duplicate rows where all feature values were identical, as they represent repeated records that do not add new information and could bias the analysis.

In [ ]:
# After noting duplicates — drop them for modeling but keep original for reference
df_clean = df.drop_duplicates()
print(f'Shape after dedup: {df_clean.shape}')

In [ ]:
# Validate binary columns only contain 0/1
binary_cols = ['HighBP','HighChol','CholCheck','Smoker','Stroke',
               'HeartDiseaseorAttack','PhysActivity','Fruits','Veggies',
               'HvyAlcoholConsump','AnyHealthcare','NoDocbcCost','DiffWalk','Sex']

for col in binary_cols:
    vals = sorted(df[col].unique())
    if vals != [0, 1]:
        print(f'WARNING: {col} has unexpected values: {vals}')
print('Binary column validation done.')

## 3. Target Variable — Class Imbalance

In [ ]:
counts = df_clean['Diabetes_binary'].value_counts()
pcts   = df_clean['Diabetes_binary'].value_counts(normalize=True) * 100

print('Class Distribution:')
print(pd.DataFrame({'Count': counts, 'Percent': pcts.round(2)}))

fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(['No Diabetes (0)', 'Prediabetes/Diabetes (1)'],
       counts.values, color=['steelblue', 'coral'], edgecolor='white')
for i, (c, p) in enumerate(zip(counts.values, pcts.values)):
    ax.text(i, c + 500, f'{c:,}\n({p:.1f}%)', ha='center', fontsize=10)
ax.set_title('Class Distribution — Binary Dataset', fontsize=13)
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

print(f'\nImbalance ratio: {counts[0]/counts[1]:.1f}:1 (no-diabetes : diabetes)')

> **Key Observation:** The dataset is imbalanced (~85% no diabetes, ~15% prediabetes/diabetes). 
> This means accuracy alone will be misleading — models need to be evaluated on precision, recall, F1, and ROC-AUC.  
> Techniques like **SMOTE** or **class_weight** should be considered at the preprocessing stage.

## 4. Continuous Features — Distributions & Outliers

In [ ]:
continuous = ['BMI', 'MentHlth', 'PhysHlth']

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, col in zip(axes, continuous):

    sns.histplot(
        data=df_clean,
        x=col,
        hue='Diabetes_binary',
        kde=False,  
        stat='density',
        common_norm=False,
        ax=ax,
        palette={0: 'steelblue', 1: 'coral'},
        alpha=0.6,
        bins=100
    )

    ax.set_title(col)
    ax.legend(title='Diabetes', labels=['No (0)', 'Yes (1)'])

plt.suptitle('Health Feature Distributions (Including Zero-Inflation)', fontsize=13)
plt.tight_layout()
plt.show()

These distributions compare health-related features between diabetic and non-diabetic groups, revealing noticeable shifts in BMI and more subtle differences in physical and mental health indicators.

In [ ]:
# BMI outlier check — clinically BMI > 60 is rare
print('BMI stats:')
print(df_clean['BMI'].describe())
print(f"\nBMI > 60: {(df_clean['BMI'] > 60).sum()} rows")
print(f"BMI > 80: {(df_clean['BMI'] > 80).sum()} rows")

In [ ]:
df_clean = df_clean.copy()
df_clean['Diabetes_binary'] = df_clean['Diabetes_binary'].astype(int)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, col in zip(axes, continuous):

    sns.boxplot(
        data=df_clean,
        x='Diabetes_binary',
        y=col,
        hue='Diabetes_binary',
        palette={0: 'steelblue', 1: 'coral'},
        dodge=False,
        ax=ax
    )

    ax.set_xticks([0, 1])
    ax.set_xticklabels(['No Diabetes', 'Diabetes'])
    ax.legend_.remove()
    ax.set_title(f'{col} by Diabetes Status')

plt.tight_layout()
plt.show()

## 5. Ordinal Features — Distributions

In [ ]:
ordinal = {'GenHlth': 'General Health (1=Excellent → 5=Poor)',
           'Age':     'Age Category (1=18-24 → 13=80+)',
           'Education': 'Education Level (1=None → 6=College grad)',
           'Income':  'Income Level (1=<$10k → 8=$75k+)'}

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
axes = axes.flatten()

for ax, (col, label) in zip(axes, ordinal.items()):
    # Diabetes rate per level
    rate = df_clean.groupby(col)['Diabetes_binary'].mean() * 100
    counts_all = df_clean[col].value_counts().sort_index()
    ax2 = ax.twinx()
    ax.bar(rate.index, rate.values, color='coral', alpha=0.7, label='Diabetes Rate (%)')
    ax2.plot(counts_all.index, counts_all.values, 'o--', color='steelblue', label='N')
    ax.set_xlabel(col)
    ax.set_ylabel('Diabetes Rate (%)', color='coral')
    ax2.set_ylabel('Count', color='steelblue')
    ax.set_title(label)

plt.suptitle('Diabetes Rate & Sample Count Across Ordinal Features', fontsize=13)
plt.tight_layout()
plt.show()

This plot shows how diabetes prevalence changes across ordinal feature levels (coral bars), while the blue dashed line shows the number of observations in each category. This helps distinguish true patterns in diabetes risk from noise caused by small sample sizes in certain groups.

> **Key Observations:**  
> - Diabetes rate climbs steeply with worse **GenHlth** — strongest ordinal predictor  
> - **Age** shows a clear positive trend (older = higher risk)  
> - **Income** is inversely related — higher income correlates with lower diabetes prevalence (access to care, lifestyle factors)  
> - **Education** follows a similar inverse pattern

## 6. Binary Features — Diabetes Rate Comparison

In [ ]:
binary_features = ['HighBP','HighChol','CholCheck','Smoker','Stroke',
                   'HeartDiseaseorAttack','PhysActivity','Fruits','Veggies',
                   'HvyAlcoholConsump','AnyHealthcare','NoDocbcCost','DiffWalk','Sex']

# Diabetes rate when feature=1 vs feature=0
rates = {}
for col in binary_features:
    r0 = df_clean[df_clean[col]==0]['Diabetes_binary'].mean() * 100
    r1 = df_clean[df_clean[col]==1]['Diabetes_binary'].mean() * 100
    rates[col] = {'Feature=0': r0, 'Feature=1': r1, 'Diff': r1 - r0}

rates_df = pd.DataFrame(rates).T.sort_values('Diff', ascending=False)
print(rates_df.round(2))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(rates_df))
width = 0.35
ax.bar(x - width/2, rates_df['Feature=0'], width, label='Feature = 0', color='steelblue', alpha=0.8)
ax.bar(x + width/2, rates_df['Feature=1'], width, label='Feature = 1', color='coral', alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(rates_df.index, rotation=45, ha='right')
ax.set_ylabel('Diabetes Rate (%)')
ax.set_title('Diabetes Rate by Binary Feature Value')
ax.legend()
plt.tight_layout()
plt.show()

> **Key Observations:**  
> - **HighBP, DiffWalk, Stroke, HeartDiseaseorAttack** show the largest positive difference (feature=1 → higher diabetes rate)  
> - **PhysActivity, HvyAlcoholConsump** show negative differences (feature=1 → lower diabetes rate)  
> - **Fruits/Veggies** have small protective effects  
> - These binary features align well with known clinical risk factors for Type 2 diabetes

## 7. Correlation Analysis

In [ ]:
# Correlation with target — ranked
corr_target = df_clean.corr()['Diabetes_binary'].drop('Diabetes_binary').sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 6))
colors = ['coral' if c > 0 else 'steelblue' for c in corr_target]
ax.barh(corr_target.index, corr_target.values, color=colors, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Pearson Correlation with Diabetes_binary')
ax.set_title('Feature Correlation with Target')
plt.tight_layout()
plt.show()

print(corr_target)

In [ ]:
# Full correlation heatmap
fig, ax = plt.subplots(figsize=(14, 11))
mask = np.triu(np.ones_like(df_clean.corr(), dtype=bool))
sns.heatmap(df_clean.corr(), mask=mask, annot=True, fmt='.2f',
            cmap='coolwarm', center=0, ax=ax, annot_kws={'size': 7})
ax.set_title('Feature Correlation Heatmap (Lower Triangle)', fontsize=13)
plt.tight_layout()
plt.show()

> **Key Observations:**  
> - **GenHlth, HighBP, BMI, DiffWalk, HighChol** are the top positive correlates with diabetes  
> - **Income, Education, PhysActivity** are top negative correlates  
> - **PhysHlth & GenHlth** are highly correlated with each other (0.54) — potential multicollinearity for some models  
> - **DiffWalk & PhysHlth** also correlate (0.43) — both capture physical limitation  
> Note: Pearson correlation understates nonlinear relationships; tree-based models will capture these better

## 8. Key Interaction Patterns

In [ ]:
# BMI vs Age colored by diabetes — subsample for speed
sample = df_clean.sample(5000, random_state=42)

fig, ax = plt.subplots(figsize=(9, 5))
scatter = ax.scatter(sample['Age'], sample['BMI'],
                     c=sample['Diabetes_binary'], cmap='coolwarm',
                     alpha=0.4, s=15)
plt.colorbar(scatter, ax=ax, label='Diabetes (1=Yes)')
ax.set_xlabel('Age Category (1=18-24 → 13=80+)')
ax.set_ylabel('BMI')
ax.set_title('BMI vs. Age — Colored by Diabetes Status (sample n=5,000)')
plt.tight_layout()
plt.show()

In [ ]:
# Heatmap: HighBP × HighChol → diabetes rate
pivot = df_clean.groupby(['HighBP', 'HighChol'])['Diabetes_binary'].mean().unstack() * 100

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(pivot, annot=True, fmt='.1f', cmap='Reds', ax=ax,
            xticklabels=['No High Chol', 'High Chol'],
            yticklabels=['No High BP', 'High BP'])
ax.set_title('Diabetes Rate (%) — HighBP × HighChol')
plt.tight_layout()
plt.show()

In [ ]:
# Diabetes rate by income & physical activity — socioeconomic + lifestyle interaction
pivot2 = df_clean.groupby(['Income', 'PhysActivity'])['Diabetes_binary'].mean().unstack() * 100
pivot2.columns = ['No PhysActivity', 'PhysActivity']

pivot2.plot(kind='bar', figsize=(9, 5), color=['coral','steelblue'], alpha=0.85, edgecolor='white')
plt.xlabel('Income Level (1=Lowest → 8=Highest)')
plt.ylabel('Diabetes Rate (%)')
plt.title('Diabetes Rate by Income & Physical Activity')
plt.legend(title='Physical Activity')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 9. Notes for Preprocessing & Modeling

| Issue | Finding | Recommended Action |
|---|---|---|
| **Class Imbalance** | ~86% / ~14% split | SMOTE, class_weight, evaluate with F1/ROC-AUC |
| **Duplicate rows** | ~24k duplicates | Drop before train/test split |
| **BMI outliers** | Values >60 exist | Consider capping or flagging |
| **Skewed MentHlth/PhysHlth** | Many zeros (healthy respondents) | May want to binarize or log-transform |
| **Multicollinearity** | PhysHlth–GenHlth, DiffWalk–PhysHlth | Mainly concern for Logistic Regression |
| **Feature scaling** | BMI, MentHlth, PhysHlth are continuous | Scale for LR, SVM, KNN — not needed for trees |
| **No missing values** | Dataset is clean | No imputation needed |

In [ ]:
# Quick summary stats per class for continuous features
print('Means by class:')
df_clean.groupby('Diabetes_binary')[continuous + list(ordinal.keys())].mean().round(2)